# 🚚 Pengumpulan dan Pra-pemrosesan Dataset CVRP
### Matakuliah: Kecerdasan Komputasional

---

## 📌 Ringkasan Proyek
Notebook ini memuat seluruh alur (**pipeline**) pengumpulan, kurasi, pra-pemrosesan, serta standardisasi dataset untuk permasalahan optimasi rute kendaraan berkapasitas (**Capacitated Vehicle Routing Problem / CVRP**). 

Proyek ini mempersiapkan dua kelompok data:
1. **Dataset Benchmark Internasional (CVRPLIB)**: Instans standar dari literatur ilmiah (*Augerat et al.*, *Christofides et al.*, dan *Uchoa et al.*) beserta nilai solusi terbaik yang diketahui (*Best Known Solutions / BKS*).
2. **Dataset Kasus Nyata (Kota Semarang)**: Titik logistik aktual berbasis *OpenStreetMap*, estimasi volume permintaan (*demand*) berbobot populasi demografi BPS Kota Semarang 2025, serta matriks jarak dan waktu tempuh rute jalan raya riil via *OSRM (Open Source Routing Machine)*.

Kedua sumber data distandarisasi ke dalam struktur data (*data contract*) yang seragam agar siap digunakan secara *plug-and-play* oleh algoritma metaheuristik (seperti *Genetic Algorithm*, *Particle Swarm Optimization*, *Ant Colony Optimization*, *Simulated Annealing*, dll.).

Seluruh dataset disimpan dan diorganisasi secara rapi di dalam direktori `data/`.

---

## 📑 Daftar Isi
1. [Inisialisasi Lingkungan & Import Library](#1-inisialisasi-lingkungan--import-library)
2. [Tahap 1: Dataset Benchmark Standar CVRPLIB](#2-tahap-1-dataset-benchmark-standar-cvrplib)
   - [2.1 Fungsi Loader Instans CVRPLIB](#21-fungsi-loader-instans-cvrplib)
   - [2.2 Tabel Best Known Solutions (BKS)](#22-tabel-best-known-solutions-bks)
3. [Tahap 2: Dataset Real-World Studi Kasus Kota Semarang](#3-tahap-2-dataset-real-world-studi-kasus-kota-semarang)
   - [3.1 Ekstraksi & Kurasi Titik Logistik (OSM GeoJSON)](#31-ekstraksi--kurasi-titik-logistik-osm-geojson)
   - [3.2 Data Demografi Penduduk per Kecamatan (BPS 2025)](#32-data-demografi-penduduk-per-kecamatan-bps-2025)
   - [3.3 Estimasi Permintaan Pelanggan (Demand Modeling)](#33-estimasi-permintaan-pelanggan-demand-modeling)
   - [3.4 Matriks Jarak & Waktu Tempuh Riil (OSRM API)](#34-matriks-jarak--waktu-tempuh-riil-osrm-api)
   - [3.5 Standardisasi Loader Instans Semarang](#35-standardisasi-loader-instans-semarang)
4. [Verifikasi Kompatibilitas & Ringkasan Dataset](#4-verifikasi-kompatibilitas--ringkasan-dataset)

---
## 1. Inisialisasi Lingkungan & Import Library

Pada bagian ini, dependensi dipersiapkan dan diimpor secara terpusat. Kami mengonfigurasi direktori penyimpanan data ke folder `data/` (`data/cvrplib/` untuk dataset benchmark dan `data/semarang/` untuk studi kasus nyata) agar alur kerja terstruktur dengan baik.

In [1]:
# Instalasi pustaka pendukung (uncomment jika dijalankan pada Google Colab atau environment baru)
# %pip install -q vrplib geopandas geopy requests pandas numpy

In [2]:
import os
import sys
import json
from pathlib import Path
import warnings
import requests

import numpy as np
import pandas as pd

# Konfigurasi direktori data utama (data/)
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
CVRP_DIR = DATA_DIR / "cvrplib"
SEMARANG_DIR = DATA_DIR / "semarang"

# Pastikan seluruh folder target tersedia
CVRP_DIR.mkdir(parents=True, exist_ok=True)
SEMARANG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Direktori Data Aktif : {DATA_DIR}")
print(f"Direktori CVRPLIB    : {CVRP_DIR}")
print(f"Direktori Semarang   : {SEMARANG_DIR}")

Direktori Data Aktif : data
Direktori CVRPLIB    : data/cvrplib
Direktori Semarang   : data/semarang

---
## 2. Tahap 1: Dataset Benchmark Standar CVRPLIB

Dataset CVRPLIB menyediakan instans benchmark standar internasional yang banyak digunakan dalam literatur riset optimasi kombinatorial. Setiap instans mencakup:
- File `.vrp`: Memuat metadata kapasitas kendaraan ($Q$), koordinat depot dan pelanggan, serta permintaan masing-masing titik.
- File `.sol`: Memuat rute optimal dan nilai objektif optimal (*Best Known Solution / BKS*).

### Konvensi Pembulatan Jarak Euclidean CVRPLIB:
- Untuk instans set **A** (*Augerat et al.*) dan set **X** (*Uchoa et al.*), nilai jarak Euclidean antar titik dibulatkan ke bilangan bulat terdekat (*integer rounding*):
  $$d_{ij} = \left\lfloor \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2} + 0.5 \right\rfloor$$
- Untuk instans set **CMT** (*Christofides, Mingozzi, Toth*), nilai jarak riil (*floating point*) dipertahankan tanpa pembulatan.

### 2.1 Fungsi Loader Instans CVRPLIB
Fungsi `load_cvrp_instance()` membaca file `.vrp` dari folder `data/cvrplib`, menghitung matriks jarak Euclidean sesuai aturan pembulatan masing-masing rumpun instans, dan menyusun dictionary instans dengan struktur standar.

In [3]:
import vrplib

def load_cvrp_instance(name: str, folder: Path | str = CVRP_DIR) -> dict:
    """
    Membaca dan memproses instans CVRP dari file CVRPLIB (.vrp).
    
    Parameters:
        name (str): Nama instans (misal: 'A-n32-k5', 'CMT1', 'X-n101-k25')
        folder (Path | str): Direktori tempat file .vrp berada (default: data/cvrplib)
        
    Returns:
        dict: Struktur instans berisi name, node_coord, demand, capacity, depot, dan edge_weight.
    """
    folder = Path(folder)
    vrp_path = folder / f"{name}.vrp"
    
    if not vrp_path.exists():
        raise FileNotFoundError(f"File instans tidak ditemukan: {vrp_path}")
        
    # Membaca isi file .vrp
    inst = vrplib.read_instance(str(vrp_path))

    coords = inst["node_coord"]
    # Hitung matriks jarak Euclidean antar seluruh pasangan node
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    dist_matrix = np.sqrt(np.sum(diff ** 2, axis=-1))

    # Terapkan konvensi pembulatan CVRPLIB
    if name.startswith("A-") or name.startswith("X-"):
        dist_matrix = np.round(dist_matrix)
    # Untuk CMT*, biarkan tanpa pembulatan (float)

    inst["edge_weight"] = dist_matrix
    return inst

# --- Verifikasi Penggunaan Instans CVRPLIB ---
sample_name = "A-n32-k5"
data_a32 = load_cvrp_instance(sample_name)
sol_path = CVRP_DIR / f"{sample_name}.sol"
sol_a32 = vrplib.read_solution(str(sol_path)) if sol_path.exists() else {"cost": "N/A"}

print(f"=== Verifikasi Instans: {sample_name} ===")
print("Kapasitas kendaraan (Q)     :", data_a32["capacity"])
print("Jumlah total node (V)       :", len(data_a32["demand"]))
print("Permintaan per node         :", data_a32["demand"])
print("Nilai optimal jarak (.sol)  :", sol_a32.get("cost"))
print("Ukuran matriks bobot sisi   :", data_a32["edge_weight"].shape)

=== Verifikasi Instans: A-n32-k5 ===
Kapasitas kendaraan (Q)     : 100
Jumlah total node (V)       : 32
Permintaan per node         : [ 0 19 21  6 19  7 12 16  6 16  8 14 21 16  3 22 18 19  1 24  8 12  4  8
 24 24  2 20 15  2 14  9]
Nilai optimal jarak (.sol)  : 784
Ukuran matriks bobot sisi   : (32, 32)

### 2.2 Tabel Best Known Solutions (BKS)
Tabel referensi berikut merangkum nilai solusi terbaik (*Best Known Solution*) dari literatur ilmiah rujukan:
- **Uchoa et al. (2017)** *New Benchmark Instances for the Capacitated Vehicle Routing Problem* (Tabel 1 & Tabel A.9)
- **Kramer et al. (2014)** *A math-heuristic approach for the split delivery vehicle routing problem* (Tabel 2)

In [4]:
# Daftar Best Known Solutions (BKS) untuk pengujian dan evaluasi algoritma
bks_data = [
    {"instance": "A-n32-k5",   "objektif": "jarak", "nilai": 784.0,    "sumber": "Uchoa dkk. (2017) Tabel A.9"},
    {"instance": "A-n45-k7",   "objektif": "jarak", "nilai": 1146.0,   "sumber": "Uchoa dkk. (2017) Tabel A.9"},
    {"instance": "A-n60-k9",   "objektif": "jarak", "nilai": 1354.0,   "sumber": "Uchoa dkk. (2017) Tabel A.9"},
    {"instance": "A-n80-k10",  "objektif": "jarak", "nilai": 1763.0,   "sumber": "Uchoa dkk. (2017) Tabel A.9"},
    {"instance": "CMT1",       "objektif": "FCVRP", "nilai": 751.11,   "sumber": "Kramer dkk. (2014) Tabel 2"},
    {"instance": "CMT2",       "objektif": "FCVRP", "nilai": 1172.62,  "sumber": "Kramer dkk. (2014) Tabel 2"},
    {"instance": "CMT3",       "objektif": "FCVRP", "nilai": 1147.83,  "sumber": "Kramer dkk. (2014) Tabel 2"},
    {"instance": "CMT12",      "objektif": "FCVRP", "nilai": 1174.02,  "sumber": "Kramer dkk. (2014) Tabel 2"},
    {"instance": "X-n101-k25", "objektif": "jarak", "nilai": 27591.0,  "sumber": "Uchoa dkk. (2017) Tabel 1"},
    {"instance": "X-n110-k13", "objektif": "jarak", "nilai": 14971.0,  "sumber": "Uchoa dkk. (2017) Tabel 1"},
    {"instance": "X-n120-k6",  "objektif": "jarak", "nilai": 13332.0,  "sumber": "Uchoa dkk. (2017) Tabel 1"},
]

df_bks = pd.DataFrame(bks_data)
bks_path = DATA_DIR / "bks.csv"
df_bks.to_csv(bks_path, index=False)

print(f"File bks.csv berhasil disimpan di: {bks_path}")
display(df_bks)

File bks.csv berhasil disimpan di: data/bks.csv

,instance,objektif,nilai,sumber
0,A-n32-k5,jarak,784.00,Uchoa dkk. (2017) Tabel A.9
1,A-n45-k7,jarak,1146.00,Uchoa dkk. (2017) Tabel A.9
2,A-n60-k9,jarak,1354.00,Uchoa dkk. (2017) Tabel A.9
3,A-n80-k10,jarak,1763.00,Uchoa dkk. (2017) Tabel A.9
4,CMT1,FCVRP,751.11,Kramer dkk. (2014) Tabel 2
5,CMT2,FCVRP,1172.62,Kramer dkk. (2014) Tabel 2
6,CMT3,FCVRP,1147.83,Kramer dkk. (2014) Tabel 2
7,CMT12,FCVRP,1174.02,Kramer dkk. (2014) Tabel 2
8,X-n101-k25,jarak,27591.00,Uchoa dkk. (2017) Tabel 1
9,X-n110-k13,jarak,14971.00,Uchoa dkk. (2017) Tabel 1


---
## 3. Tahap 2: Dataset Real-World Studi Kasus Kota Semarang

Dataset studi kasus nyata dikembangkan untuk merepresentasikan skenario distribusi logistik di **Kota Semarang**:
- **Depot Sentral**: Depot Logistik Ronggowarsito (Kecamatan Semarang Utara, dekat koridor industri & Pelabuhan Tanjung Emas).
- **Titik Pelanggan**: 30 titik distribusi/pasar yang tersebar merata di 16 kecamatan administratif Kota Semarang.
- **Karakteristik Jalur**: Menggunakan jarak tempuh dan waktu tempuh jaringan jalan raya riil via *OSRM (Open Source Routing Machine)*.

### 3.1 Ekstraksi & Kurasi Titik Logistik (OSM GeoJSON)
Pipeline kurasi data geospasial:
1. **Proyeksi Geospasial**: Proyeksi ke UTM Zona 49S (`EPSG:32749`) untuk menghitung titik pusat (*centroid*) dengan satuan meter planar yang akurat.
2. **Filter Wilayah Administratif**: Menyaring batas koordinat latitude $[-7.12, -6.93]$ dan longitude $[110.27, 110.50]$.
3. **Filter Jarak Minimum**: Menghapus titik yang berjarak kurang dari 300 meter untuk menghindari duplikasi kompleks pasar yang berdekatan.
4. **Validasi Kecamatan**: Menggunakan *reverse geocoding* (Nominatim) untuk memvalidasi lokasi ke dalam 16 kecamatan resmi.
5. **Sampling Merata**: Memilih 30 titik dengan metode *round-robin* antar kecamatan, lalu menambahkan 1 Depot Pusat.

In [5]:
titik_path = SEMARANG_DIR / "titik.csv"
geojson_path = SEMARANG_DIR / "export.geojson"

# Jika file titik.csv sudah tersedia, muat langsung untuk efisiensi eksekusi
if titik_path.exists():
    df_titik = pd.read_csv(titik_path)
    print(f"Data titik logistik dimuat dari cache: {titik_path}")
    print(f"Total titik tersimpan: {len(df_titik)} (1 Depot + {len(df_titik)-1} Pelanggan)")
    print("\nSebaran titik per kecamatan:")
    print(df_titik["kecamatan"].value_counts())
else:
    import geopandas as gpd
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter

    # 1. Baca GeoJSON
    gdf = gpd.read_file(geojson_path)

    # 2. Proyeksi ke UTM 49S untuk centroid akurat
    gdf_utm = gdf.to_crs(epsg=32749)
    centroids_utm = gdf_utm.geometry.centroid
    centroids_wgs84 = centroids_utm.to_crs(epsg=4326)

    gdf["lon"] = centroids_wgs84.x
    gdf["lat"] = centroids_wgs84.y
    gdf["x_m"] = centroids_utm.x
    gdf["y_m"] = centroids_utm.y

    if "name" not in gdf.columns:
        gdf["name"] = [f"Titik Logistik {i+1}" for i in range(len(gdf))]
    else:
        gdf["name"] = gdf["name"].fillna("Tanpa Nama")

    # 3. Filter batas geografis Kota Semarang
    gdf = gdf[
        (gdf["lat"] >= -7.12) & (gdf["lat"] <= -6.93) &
        (gdf["lon"] >= 110.27) & (gdf["lon"] <= 110.50)
    ].copy()

    # Hapus duplikat nama
    gdf = gdf.drop_duplicates(subset=["name"]).reset_index(drop=True)

    # Filter jarak minimum 300 meter
    keep_indices = []
    coords_m = gdf[["x_m", "y_m"]].values
    for i in range(len(gdf)):
        if not keep_indices:
            keep_indices.append(i)
        else:
            dists = np.linalg.norm(coords_m[keep_indices] - coords_m[i], axis=1)
            if np.min(dists) >= 300.0:
                keep_indices.append(i)

    gdf = gdf.iloc[keep_indices].copy()

    # 4. Deteksi & Validasi 16 Kecamatan Resmi Kota Semarang
    KECAMATAN_SEMARANG = {
        "Banyumanik", "Candisari", "Gajahmungkur", "Gayamsari",
        "Genuk", "Gunungpati", "Mijen", "Ngaliyan",
        "Pedurungan", "Semarang Barat", "Semarang Selatan", "Semarang Tengah",
        "Semarang Timur", "Semarang Utara", "Tembalang", "Tugu"
    }

    print(f"Mendeteksi kecamatan untuk {len(gdf)} kandidat titik bersih...")
    geolocator = Nominatim(user_agent="green_cvrp_semarang_v2")
    geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1)

    def get_valid_kecamatan(row):
        try:
            loc = geocode((row["lat"], row["lon"]), language="id")
            if loc and "address" in loc.raw:
                addr = loc.raw["address"]
                for val in addr.values():
                    clean_val = str(val).replace("Kecamatan ", "").strip()
                    if clean_val in KECAMATAN_SEMARANG:
                        return clean_val
        except Exception:
            pass
        return "Tidak Valid"

    gdf["kecamatan"] = gdf.apply(get_valid_kecamatan, axis=1)
    gdf_valid = gdf[gdf["kecamatan"] != "Tidak Valid"].copy()

    # 5. Round-Robin sampling untuk memilih 30 titik pelanggan secara merata
    selected_rows = []
    grouped = {kec: group.index.tolist() for kec, group in gdf_valid.groupby("kecamatan")}

    while len(selected_rows) < 30 and any(grouped.values()):
        for kec in sorted(grouped.keys()):
            if grouped[kec] and len(selected_rows) < 30:
                selected_rows.append(grouped[kec].pop(0))

    gdf_final = gdf_valid.loc[selected_rows].reset_index(drop=True)

    # 6. Susun tabel akhir (Depot id=0 + Pelanggan id=1..n)
    df_pelanggan = pd.DataFrame({
        "id": range(1, len(gdf_final) + 1),
        "nama": gdf_final["name"].values,
        "lat": gdf_final["lat"].values,
        "lon": gdf_final["lon"].values,
        "kecamatan": gdf_final["kecamatan"].values
    })

    df_depot = pd.DataFrame([{
        "id": 0,
        "nama": "Depot Logistik Ronggowarsito",
        "lat": -6.96345,
        "lon": 110.42785,
        "kecamatan": "Semarang Utara"
    }])

    df_titik = pd.concat([df_depot, df_pelanggan], ignore_index=True)
    df_titik.to_csv(titik_path, index=False)

    print(f"\nSelesai! Total titik tersimpan: {len(df_titik)} (1 Depot + {len(df_pelanggan)} Pelanggan)")
    print("\nSebaran titik per kecamatan:")
    print(df_titik["kecamatan"].value_counts())

Data titik logistik dimuat dari cache: data/semarang/titik.csv
Total titik tersimpan: 31 (1 Depot + 30 Pelanggan)

Sebaran titik per kecamatan:
kecamatan
Semarang Utara      3
Banyumanik          3
Candisari           3
Gayamsari           3
Genuk               2
Mijen               2
Semarang Selatan    2
Ngaliyan            2
Pedurungan          2
Semarang Barat      2
Semarang Timur      2
Semarang Tengah     2
Tembalang           2
Gajahmungkur        1
Gunungpati          1
Tugu                1
Name: count, dtype: int64

### 3.2 Data Demografi Penduduk per Kecamatan (BPS 2025)
Data jumlah penduduk per kecamatan bersumber dari publikasi **Badan Pusat Statistik (BPS) Kota Semarang 2025**. Data demografi ini digunakan sebagai dasar pembobotan estimasi kebutuhan/volume logistik per kecamatan.

In [6]:
penduduk_clean_path = SEMARANG_DIR / "penduduk_kecamatan.csv"
raw_bps_path = SEMARANG_DIR / "Jumlah Penduduk Menurut Kecamatan dan Jenis Kelamin, 2025.csv"

if raw_bps_path.exists():
    # 1. Baca file mentah BPS sambil melewati 4 baris header bertingkat
    df_penduduk = pd.read_csv(
        raw_bps_path,
        skiprows=4,
        names=["kecamatan", "laki_laki", "perempuan"],
    )
    # 2. Hapus baris total 'Kota Semarang' dan bersihkan spasi
    df_penduduk["kecamatan"] = df_penduduk["kecamatan"].str.strip()
    df_penduduk = df_penduduk[df_penduduk["kecamatan"] != "Kota Semarang"].copy()

    # 3. Jumlahkan penduduk
    df_penduduk["penduduk"] = df_penduduk["laki_laki"] + df_penduduk["perempuan"]

    # 4. Samakan penulisan nama kecamatan dengan standar OSM
    mapping_nama = {
        "Gunung Pati": "Gunungpati",
        "Gajah Mungkur": "Gajahmungkur",
    }
    df_penduduk["kecamatan"] = df_penduduk["kecamatan"].replace(mapping_nama)
    df_penduduk = df_penduduk[["kecamatan", "penduduk"]]
    df_penduduk.to_csv(penduduk_clean_path, index=False)
elif penduduk_clean_path.exists():
    df_penduduk = pd.read_csv(penduduk_clean_path)
else:
    # Fallback data resmi BPS Kota Semarang 2025
    data_resmi = [
        ("Mijen", 96357), ("Gunungpati", 102429), ("Banyumanik", 144087),
        ("Gajahmungkur", 56328), ("Semarang Selatan", 61981), ("Candisari", 75396),
        ("Tembalang", 204865), ("Pedurungan", 198452), ("Genuk", 141033),
        ("Gayamsari", 70381), ("Semarang Timur", 66482), ("Semarang Utara", 117865),
        ("Semarang Tengah", 55213), ("Semarang Barat", 149357), ("Tugu", 34398),
        ("Ngaliyan", 147797)
    ]
    df_penduduk = pd.DataFrame(data_resmi, columns=["kecamatan", "penduduk"])
    df_penduduk.to_csv(penduduk_clean_path, index=False)

print(f"Data demografi kecamatan berhasil dimuat ({len(df_penduduk)} kecamatan):")
display(df_penduduk)

Data demografi kecamatan berhasil dimuat (16 kecamatan):

,kecamatan,penduduk
0,Mijen,96357
1,Gunungpati,102429
2,Banyumanik,144087
3,Gajahmungkur,56328
4,Semarang Selatan,61981
5,Candisari,75396
6,Tembalang,204865
7,Pedurungan,198452
8,Genuk,141033
9,Gayamsari,70381


### 3.3 Estimasi Permintaan Pelanggan (Demand Modeling)
Besaran permintaan logistik per titik pelanggan dimodelkan berdasarkan bobot proporsi populasi kecamatan ditambah komponen variasi acak:
$$\text{bobot}_k = \frac{\text{penduduk}_k - \text{pop}_{\min}}{\text{pop}_{\max} - \text{pop}_{\min}}$$
$$\text{demand}_i = 100 + (\text{bobot}_k \times 500) + \epsilon_i, \quad \epsilon_i \sim \mathcal{U}(0, 200)$$

- Permintaan berada pada rentang **100 kg hingga 800 kg**.
- Titik depot sentral ($id = 0$) secara mutlak memiliki permintaan $\text{demand}_0 = 0$.

In [7]:
# Baca data titik dan populasi kecamatan
df_titik = pd.read_csv(SEMARANG_DIR / "titik.csv")
df_penduduk = pd.read_csv(SEMARANG_DIR / "penduduk_kecamatan.csv")

# Gabungkan data berdasarkan nama kecamatan
df_merged = df_titik.merge(df_penduduk, on="kecamatan", how="left")

# Tetapkan seed acak agar hasil replikatif
np.random.seed(42)

# Normalisasi bobot penduduk (0 s.d. 1)
min_pop = df_merged["penduduk"].min()
max_pop = df_merged["penduduk"].max()
bobot = (df_merged["penduduk"] - min_pop) / (max_pop - min_pop)

# Bangkitkan demand 100 - 800 kg yang dipengaruhi bobot kecamatan + variasi acak
base_demand = 100 + (bobot * 500) + np.random.randint(0, 200, size=len(df_merged))
df_merged["demand_kg"] = np.round(base_demand).astype(int)

# Set demand depot (id == 0) menjadi 0
df_merged.loc[df_merged["id"] == 0, "demand_kg"] = 0

# Simpan ke demand_kg.csv
demand_path = SEMARANG_DIR / "demand_kg.csv"
df_demand_out = df_merged[["id", "nama", "kecamatan", "demand_kg"]]
df_demand_out.to_csv(demand_path, index=False)

print(f"File demand_kg.csv berhasil dibuat di: {demand_path}")
print(f"- Total Titik Pelanggan : {len(df_demand_out[df_demand_out['id'] > 0])}")
print(f"- Total Demand Muatan   : {df_demand_out['demand_kg'].sum():,} kg")
print(f"- Rata-rata Demand      : {df_demand_out[df_demand_out['id'] > 0]['demand_kg'].mean():.2f} kg")
print(f"- Rentang Demand        : [{df_demand_out[df_demand_out['id'] > 0]['demand_kg'].min()} kg - {df_demand_out['demand_kg'].max()} kg]")
display(df_demand_out.head(10))

File demand_kg.csv berhasil dibuat di: data/semarang/demand_kg.csv
- Total Titik Pelanggan : 30
- Total Demand Muatan   : 13,007 kg
- Rata-rata Demand      : 433.57 kg
- Rentang Demand        : [151 kg - 755 kg]

,id,nama,kecamatan,demand_kg
0,0,Depot Logistik Ronggowarsito,Semarang Utara,0
1,1,Pasar Jatingaleh,Banyumanik,524
2,2,Pasar Kagok,Candisari,319
3,3,Pasar Sampangan Baru,Gajahmungkur,243
4,4,Pasar Gayamsari,Gayamsari,311
5,5,Pasar Bangetayu,Genuk,484
6,6,Pasar Mijen,Mijen,361
7,7,Pasar Jrakah,Ngaliyan,503
8,8,Pasar Unggas Penggaron,Pedurungan,660
9,9,Pasar Karangayu,Semarang Barat,516


### 3.4 Matriks Jarak & Waktu Tempuh Riil (OSRM API)
Jarak dan waktu tempuh antar-titik dihitung menggunakan **OSRM (Open Source Routing Machine) Table API** dengan profil berkendara (*driving*):
- **Matriks Jarak (`jarak_m.csv`)**: Jarak rute jalan raya dalam satuan **meter**.
- **Matriks Waktu (`waktu_s.csv`)**: Estimasi waktu perjalanan dalam satuan **detik**.

In [8]:
jarak_path = SEMARANG_DIR / "jarak_m.csv"
waktu_path = SEMARANG_DIR / "waktu_s.csv"

df_titik_sorted = pd.read_csv(SEMARANG_DIR / "titik.csv").sort_values("id")
coords_str = ";".join([f"{row['lon']},{row['lat']}" for _, row in df_titik_sorted.iterrows()])

# Gunakan data tersimpan jika sudah ada, atau panggil API OSRM jika belum ada
if jarak_path.exists() and waktu_path.exists():
    dist_matrix = pd.read_csv(jarak_path, header=None).values
    dur_matrix = pd.read_csv(waktu_path, header=None).values
    print("Matriks jarak_m.csv dan waktu_s.csv berhasil dimuat dari cache lokal!")
else:
    print("Memanggil OSRM Table API untuk 31 titik...")
    url = f"http://router.project-osrm.org/table/v1/driving/{coords_str}?annotations=distance,duration"
    try:
        response = requests.get(url, timeout=30)
        data = response.json()
        if data.get("code") == "Ok":
            dist_matrix = np.array(data["distances"])  # satuan meter
            dur_matrix = np.array(data["durations"])   # satuan detik
            pd.DataFrame(dist_matrix).to_csv(jarak_path, index=False, header=False)
            pd.DataFrame(dur_matrix).to_csv(waktu_path, index=False, header=False)
            print("Matriks jarak_m.csv dan waktu_s.csv berhasil disimpan dari OSRM API!")
        else:
            print("Gagal mengambil dari OSRM:", data)
    except Exception as e:
        print(f"Peringatan: Gagal menghubungi server OSRM ({e})")

print(f"Dimensi Matriks Jarak    : {dist_matrix.shape} (meter)")
print(f"Dimensi Matriks Waktu    : {dur_matrix.shape} (detik)")
print(f"Jarak Terjauh Antartitik : {dist_matrix.max() / 1000:.2f} km")
print(f"Jarak Rata-rata          : {dist_matrix[dist_matrix > 0].mean() / 1000:.2f} km")

Matriks jarak_m.csv dan waktu_s.csv berhasil dimuat dari cache lokal!
Dimensi Matriks Jarak    : (31, 31) (meter)
Dimensi Matriks Waktu    : (31, 31) (detik)
Jarak Terjauh Antartitik : 27.84 km
Jarak Rata-rata          : 12.43 km

### 3.5 Standardisasi Loader Instans Semarang
Fungsi `load_semarang_instance()` menyusun dictionary instans studi kasus Semarang dengan *schema* yang sama persis seperti instans CVRPLIB:
- `name`: Identifikasi instans (`"Semarang-Case-Study"`).
- `node_coord`: Koordinat lokasi $[lon, lat]$.
- `demand`: Array bobot permintaan masing-masing node.
- `capacity`: Kapasitas armada truk pengangkut ($Q$).
- `depot`: Indeks lokasi depot (node 0).
- `edge_weight`: Matriks jarak jalan riil (meter).

In [9]:
def load_semarang_instance(kapasitas_truk_kg: int = 3000, folder: Path | str = SEMARANG_DIR) -> dict:
    """
    Memuat instans CVRP studi kasus Kota Semarang.
    
    Parameters:
        kapasitas_truk_kg (int): Kapasitas angkut kendaraan (default: 3000 kg).
        folder (Path | str): Direktori data semarang (default: data/semarang).
        
    Returns:
        dict: Struktur data instans terstandarisasi.
    """
    folder = Path(folder)
    df_pts = pd.read_csv(folder / "titik.csv").sort_values("id")
    df_dem = pd.read_csv(folder / "demand_kg.csv").sort_values("id")
    jarak_m = pd.read_csv(folder / "jarak_m.csv", header=None).values

    inst = {
        "name": "Semarang-Case-Study",
        "node_coord": df_pts[["lon", "lat"]].values,
        "demand": df_dem["demand_kg"].values,
        "capacity": kapasitas_truk_kg,
        "depot": np.array([0]),
        "edge_weight": jarak_m
    }
    return inst

# --- CARA PENGGUNAAN ---
# 1. Panggil fungsi (misal: kapasitas truk sedang 3000 kg)
data_semarang = load_semarang_instance(kapasitas_truk_kg=3000)

# 2. Akses komponen datanya sama persis seperti saat mengakses data CVRPLIB:
koordinat = data_semarang["node_coord"]
permintaan = data_semarang["demand"]
kapasitas_Q = data_semarang["capacity"]
matriks_jarak = data_semarang["edge_weight"]

print("=== Ringkasan Instans Studi Kasus Kota Semarang ===")
print("Nama Instans                     :", data_semarang["name"])
print("Total Node (1 Depot + Pelanggan) :", len(permintaan))
print("Indeks Depot                     :", data_semarang["depot"][0])
print("Kapasitas Kendaraan (Q)          :", kapasitas_Q, "kg")
print("Total Kebutuhan Angkut           :", permintaan.sum(), "kg")
print("Ukuran Matriks Jarak             :", matriks_jarak.shape)

=== Ringkasan Instans Studi Kasus Kota Semarang ===
Nama Instans                     : Semarang-Case-Study
Total Node (1 Depot + Pelanggan) : 31
Indeks Depot                     : 0
Kapasitas Kendaraan (Q)          : 3000 kg
Total Kebutuhan Angkut           : 13007 kg
Ukuran Matriks Jarak             : (31, 31)

---
## 4. Verifikasi Kompatibilitas & Ringkasan Dataset

Untuk memastikan bahwa modul algoritma optimasi rute (seperti *Genetic Algorithm*, *Particle Swarm Optimization*, *Ant Colony Optimization*, dll.) dapat langsung dijalankan tanpa modifikasi antarmuka, berikut verifikasi keselarasan antarmuka (*interface contract*) antara dataset benchmark CVRPLIB dan dataset nyata Kota Semarang.

In [10]:
# Verifikasi integritas schema antarmuka
schema_keys = ["name", "node_coord", "demand", "capacity", "depot", "edge_weight"]

instances_to_compare = [
    ("CVRPLIB Benchmark (A-n32-k5)", data_a32),
    ("Studi Kasus Kota Semarang", data_semarang)
]

summary_rows = []
for label, inst in instances_to_compare:
    has_all_keys = all(k in inst for k in schema_keys)
    summary_rows.append({
        "Kelompok Instans": label,
        "Nama": inst.get("name"),
        "Jumlah Node": len(inst["demand"]),
        "Kapasitas (Q)": inst["capacity"],
        "Total Demand": int(np.sum(inst["demand"])),
        "Dimensi Bobot": f"{inst['edge_weight'].shape[0]} × {inst['edge_weight'].shape[1]}",
        "Kesesuaian Schema": "✅ Valid & Kompatibel" if has_all_keys else "❌ Tidak Lengkap"
    })

df_summary = pd.DataFrame(summary_rows)
print("=== Ringkasan Validasi Schema Antarmuka Dataset ===")
display(df_summary)

=== Ringkasan Validasi Schema Antarmuka Dataset ===

,Kelompok Instans,Nama,Jumlah Node,Kapasitas (Q),Total Demand,Dimensi Bobot,Kesesuaian Schema
0,CVRPLIB Benchmark (A-n32-k5),A-n32-k5,32,100,410,32 × 32,✅ Valid & Kompatibel
1,Studi Kasus Kota Semarang,Semarang-Case-Study,31,3000,13007,31 × 31,✅ Valid & Kompatibel
